In [2]:
pip install lxml_html_clean

Note: you may need to restart the kernel to use updated packages.


In [1]:
import logging
import time

import pandas as pd
import requests
import trafilatura
from lxml import html

# Matikan warning 
logging.getLogger("urllib3").setLevel(logging.ERROR)
logging.getLogger("trafilatura").setLevel(logging.ERROR)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# ================================================================
# TAHAP 1: Mengumpulkan URL Artikel dari Halaman Indeks
# Menggunakan requests + lxml (XPath)
# ================================================================
def get_article_links(base_index_url, label, target_count=100):
    links = []
    page = 1

    print(f"\n[{label}] Mengumpulkan URL dari halaman indeks...")

    while len(links) < target_count:
        url = f"{base_index_url}?page={page}"
        res = requests.get(url, headers=HEADERS, timeout=10)

        if res.status_code != 200:
            print(f"  Halaman {page}: gagal (status {res.status_code})")
            break

        # Parse HTML menggunakan lxml
        tree = html.fromstring(res.text)

        # Gunakan XPath untuk mencari link di dalam tag <article>
        # XPath: //article//a/@href → ambil semua atribut href dari tag <a>
        #        yang berada di dalam tag <article>
        hrefs = tree.xpath("//article//a/@href")

        if not hrefs:
            print(f"  Halaman {page}: tidak ada artikel ditemukan")
            break

        for href in hrefs:
            # Filter: hanya ambil link artikel detik (mengandung /d-)
            if "detik.com" in href and "/d-" in href and href not in links:
                links.append(href)
                if len(links) >= target_count:
                    break

        print(f"  Halaman {page}: terkumpul {len(links)} link")
        page += 1
        time.sleep(0.5)  # Jeda biar tidak diblokir server

    print(f"[{label}] Total: {len(links)} URL artikel")
    return links


# ================================================================
# TAHAP 2: Mengekstrak Konten Artikel
# Menggunakan trafilatura
# ================================================================
def ekstrak_konten(url):
    try:
        downloaded = trafilatura.fetch_url(url)
        if downloaded is None:
            return ""

        text = trafilatura.extract(downloaded)
        return text if text else ""

    except Exception:
        return ""


# ================================================================
# MAIN: Alur Utama Program
# ================================================================
def main():
    TARGET_PER_KATEGORI = 100

    # Tahap 1: Gathering URL 
    print("=" * 60)
    print("TAHAP 1: Mengumpulkan URL Artikel")
    print("=" * 60)

    sport_links = get_article_links(
        "https://sport.detik.com/indeks",
        label="SPORT",
        target_count=TARGET_PER_KATEGORI,
    )

    finance_links = get_article_links(
        "https://finance.detik.com/indeks",
        label="FINANCE",
        target_count=TARGET_PER_KATEGORI,
    )

    # Tahap 2: Ekstrak Konten 
    print("\n" + "=" * 60)
    print("TAHAP 2: Mengekstrak Konten Artikel")
    print("=" * 60)

    data_rows = []
    current_id = 1

    print(f"\n[SPORT] Mengekstrak {len(sport_links)} artikel...")
    for i, link in enumerate(sport_links, 1):
        content = ekstrak_konten(link)
        if content:
            data_rows.append(
                {"id": current_id, "isi_berita": content, "label": "sport"}
            )
            current_id += 1
            print(f"  [{i}/{len(sport_links)}] OK")
        else:
            print(f"  [{i}/{len(sport_links)}] SKIP")
        time.sleep(0.3)

    print(f"\n[FINANCE] Mengekstrak {len(finance_links)} artikel...")
    for i, link in enumerate(finance_links, 1):
        content = ekstrak_konten(link)
        if content:
            data_rows.append(
                {"id": current_id, "isi_berita": content, "label": "finance"}
            )
            current_id += 1
            print(f"  [{i}/{len(finance_links)}] OK")
        else:
            print(f"  [{i}/{len(finance_links)}] SKIP")
        time.sleep(0.3)

    # --- Tahap 3: Simpan ---
    print("\n" + "=" * 60)
    print("TAHAP 3: Menyimpan Data")
    print("=" * 60)

    df = pd.DataFrame(data_rows)

    df.to_csv("data_berita_detik.csv", index=False, encoding="utf-8")
    print("  Tersimpan: data_berita_detik.csv")

    df.to_excel("data_berita_detik.xlsx", index=False)
    print("  Tersimpan: data_berita_detik.xlsx")

    # Ringkasan
    print("\n" + "=" * 60)
    print("RINGKASAN")
    print("=" * 60)

    sport_count = len(df[df["label"] == "sport"])
    finance_count = len(df[df["label"] == "finance"])

    print(f"  Total data      : {len(df)}")
    print(f"  Artikel Sport   : {sport_count}")
    print(f"  Artikel Finance : {finance_count}")
    print(f"  Kolom           : {list(df.columns)}")
    print(f"\nPreview:")
    print(df.head())
    print("\nSelesai!")


if __name__ == "__main__":
    main()

TAHAP 1: Mengumpulkan URL Artikel

[SPORT] Mengumpulkan URL dari halaman indeks...
  Halaman 1: terkumpul 20 link
  Halaman 2: terkumpul 40 link
  Halaman 3: terkumpul 60 link
  Halaman 4: terkumpul 80 link
  Halaman 5: terkumpul 99 link
  Halaman 6: terkumpul 100 link
[SPORT] Total: 100 URL artikel

[FINANCE] Mengumpulkan URL dari halaman indeks...
  Halaman 1: terkumpul 20 link
  Halaman 2: terkumpul 39 link
  Halaman 3: terkumpul 59 link
  Halaman 4: terkumpul 79 link
  Halaman 5: terkumpul 97 link
  Halaman 6: terkumpul 100 link
[FINANCE] Total: 100 URL artikel

TAHAP 2: Mengekstrak Konten Artikel

[SPORT] Mengekstrak 100 artikel...
  [1/100] OK
  [2/100] OK
  [3/100] OK
  [4/100] OK
  [5/100] OK
  [6/100] OK
  [7/100] OK
  [8/100] OK
  [9/100] OK
  [10/100] OK
  [11/100] OK
  [12/100] OK
  [13/100] OK
  [14/100] OK
  [15/100] OK
  [16/100] OK
  [17/100] OK
  [18/100] OK
  [19/100] OK
  [20/100] OK
  [21/100] OK
  [22/100] OK
  [23/100] OK
  [24/100] OK
  [25/100] OK
  [26/100] OK


In [1]:
# Preprocessing Data Berita Detik.com
# Membaca data_berita_detik.xlsx, membersihkan teks, lalu menyimpan hasilnya

import re
import pandas as pd

# Baca data hasil crawling
df = pd.read_excel("data_berita_detik.xlsx")
print(f"Total data: {len(df)}")

# Daftar kata tidak baku / singkatan yang akan dibuang
kata_tidak_baku = {
    'tdk', 'gak', 'ga', 'gk', 'yg', 'dgn', 'utk', 'krn', 'dg', 'dr',
    'pd', 'jg', 'lg', 'sdh', 'blm', 'tp', 'kl', 'klo', 'bs', 'bgt',
    'byk', 'smua', 'sm', 'sy', 'ak', 'gw', 'gue', 'lu', 'lo', 'aja',
    'udh', 'udah', 'emg', 'emang', 'org', 'dpt', 'stlh', 'sblm',
    'krna', 'brg', 'dri', 'dll', 'dsb', 'dst', 'tsb', 'spt', 'dmn',
    'kmn', 'gmn', 'bgmn', 'hrs', 'trs', 'thd', 'ttg', 'tgl', 'thn',
    'bln', 'jd', 'jdi', 'sdg', 'shg', 'stl', 'skrg', 'trhdp', 'dlm',
    'dkk', 'sblmnya', 'tsbt', 'sbg', 'krg', 'brp', 'dsbnya', 'msh',
    'bkn', 'blh', 'thn', 'mgkn', 'bnyk', 'smpe', 'smpai', 'kpd',
    'trmsuk', 'thp', 'dng', 'kt', 'mrk', 'sorg', 'ckp', 'bbrp'
}


def preprocess_teks(teks):
    # Hapus angka
    teks = re.sub(r'\d+', '', teks)
    # Hapus tanda baca dan karakter khusus, sisakan huruf dan spasi
    teks = re.sub(r'[^a-zA-Z\s]', '', teks)
    # Hapus spasi berlebih
    teks = re.sub(r'\s+', ' ', teks).strip()
    # Hapus kata tidak baku dan kata 1 huruf
    kata_list = teks.split()
    kata_bersih = [k for k in kata_list if k not in kata_tidak_baku and len(k) > 1]
    return ' '.join(kata_bersih)


# Terapkan preprocessing
df['isi_berita_clean'] = df['isi_berita'].astype(str).apply(preprocess_teks)

print(f"\nContoh sebelum: {df['isi_berita'].iloc[0][:100]}...")
print(f"Contoh setelah: {df['isi_berita_clean'].iloc[0][:100]}...")

# Simpan hasil preprocessing
df.to_csv("data_berita_preprocessed.csv", index=False, encoding="utf-8")
df.to_excel("data_berita_preprocessed.xlsx", index=False)
print(f"\nTersimpan: data_berita_preprocessed.csv / .xlsx")

Total data: 200

Contoh sebelum: Dua petenis Indonesia, Muhammad Rifqi Fitriadi dan Rafalentino Ali Da Costa, melangkah ke Babak Kedu...
Contoh setelah: Dua petenis Indonesia Muhammad Rifqi Fitriadi dan Rafalentino Ali Da Costa melangkah ke Babak Kedua ...

Tersimpan: data_berita_preprocessed.csv / .xlsx


In [3]:
pip install scikit-learn

  Using cached scikit_learn-1.6.1-cp39-cp39-win_amd64.whl.metadata (15 kB)
  Using cached scipy-1.13.1-cp39-cp39-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.6.1-cp39-cp39-win_amd64.whl (11.2 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.13.1-cp39-cp39-win_amd64.whl (46.2 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- -------------------------

In [4]:
# TF-IDF dan Split Training/Testing
# Membaca data_berita_preprocessed.xlsx, membuat matriks TF-IDF, lalu split 160 train + 40 test

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Baca data hasil preprocessing
df = pd.read_excel("data_berita_preprocessed.xlsx")
print(f"Total data: {len(df)}")

# Buat TF-IDF dari teks yang sudah bersih
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['isi_berita_clean'])

# Ambil nama kata unik
kata_unik = vectorizer.get_feature_names_out()
print(f"Jumlah kata unik: {len(kata_unik)}")

# Buat DataFrame TF-IDF dengan format: ID | kata1 | kata2 | ... | Label
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=kata_unik)
df_tfidf.insert(0, 'ID', df['id'].values)
df_tfidf['Label'] = df['label'].map({'sport': 1, 'finance': 2}).values

print(f"Bentuk matriks: {df_tfidf.shape}")

# Split: 160 training + 40 testing (stratified)
df_train, df_test = train_test_split(
    df_tfidf,
    test_size=40,
    train_size=160,
    random_state=42,
    stratify=df_tfidf['Label']
)

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

sport_train = len(df_train[df_train['Label'] == 1])
finance_train = len(df_train[df_train['Label'] == 2])
sport_test = len(df_test[df_test['Label'] == 1])
finance_test = len(df_test[df_test['Label'] == 2])

print(f"\nTraining: {len(df_train)} data (Sport: {sport_train}, Finance: {finance_train})")
print(f"Testing : {len(df_test)} data (Sport: {sport_test}, Finance: {finance_test})")

# Simpan
df_train.to_csv("tfidf_training.csv", index=False)
df_train.to_excel("tfidf_training.xlsx", index=False)
print(f"\nTersimpan: tfidf_training.csv / .xlsx ({df_train.shape})")

df_test.to_csv("tfidf_testing.csv", index=False)
df_test.to_excel("tfidf_testing.xlsx", index=False)
print(f"Tersimpan: tfidf_testing.csv / .xlsx ({df_test.shape})")

Total data: 200
Jumlah kata unik: 7424
Bentuk matriks: (200, 7426)

Training: 160 data (Sport: 80, Finance: 80)
Testing : 40 data (Sport: 20, Finance: 20)

Tersimpan: tfidf_training.csv / .xlsx ((160, 7426))
Tersimpan: tfidf_testing.csv / .xlsx ((40, 7426))


In [6]:
# Reduksi Dimensi dengan PCA
# Membaca tfidf_training/testing, mereduksi kolom kata unik dari ribuan menjadi 200 dimensi

import pandas as pd
from sklearn.decomposition import PCA

N_KOMPONEN = 160

# Baca data TF-IDF
df_train = pd.read_csv("tfidf_training.csv")
df_test = pd.read_csv("tfidf_testing.csv")
print(f"Training sebelum reduksi: {df_train.shape}")
print(f"Testing sebelum reduksi : {df_test.shape}")

# Pisahkan kolom fitur (tanpa ID dan Label)
fitur_train = df_train.drop(columns=['ID', 'Label']).values
fitur_test = df_test.drop(columns=['ID', 'Label']).values

# Terapkan PCA
pca = PCA(n_components=N_KOMPONEN, random_state=42)
pca_train = pca.fit_transform(fitur_train)
pca_test = pca.transform(fitur_test)

variance = sum(pca.explained_variance_ratio_) * 100
print(f"\nVariance explained: {variance:.2f}%")

# Buat DataFrame hasil: ID | PC1 | PC2 | ... | PC200 | Label
kolom_pca = [f'PC{i+1}' for i in range(N_KOMPONEN)]

df_pca_train = pd.DataFrame(pca_train, columns=kolom_pca)
df_pca_train.insert(0, 'ID', df_train['ID'].values)
df_pca_train['Label'] = df_train['Label'].values

df_pca_test = pd.DataFrame(pca_test, columns=kolom_pca)
df_pca_test.insert(0, 'ID', df_test['ID'].values)
df_pca_test['Label'] = df_test['Label'].values

print(f"Training sesudah reduksi: {df_pca_train.shape}")
print(f"Testing sesudah reduksi : {df_pca_test.shape}")

# Simpan
df_pca_train.to_csv("tfidf_pca_training.csv", index=False)
df_pca_train.to_excel("tfidf_pca_training.xlsx", index=False)
print(f"\nTersimpan: tfidf_pca_training.csv / .xlsx")

df_pca_test.to_csv("tfidf_pca_testing.csv", index=False)
df_pca_test.to_excel("tfidf_pca_testing.xlsx", index=False)
print(f"Tersimpan: tfidf_pca_testing.csv / .xlsx")

Training sebelum reduksi: (160, 7426)
Testing sebelum reduksi : (40, 7426)

Variance explained: 100.00%
Training sesudah reduksi: (160, 162)
Testing sesudah reduksi : (40, 162)

Tersimpan: tfidf_pca_training.csv / .xlsx
Tersimpan: tfidf_pca_testing.csv / .xlsx


In [7]:
# Reduksi Dimensi dengan SVD (TruncatedSVD)
# Membaca tfidf_training/testing, mereduksi kolom kata unik dari ribuan menjadi 200 dimensi

import pandas as pd
from sklearn.decomposition import TruncatedSVD

N_KOMPONEN = 160

# Baca data TF-IDF
df_train = pd.read_csv("tfidf_training.csv")
df_test = pd.read_csv("tfidf_testing.csv")
print(f"Training sebelum reduksi: {df_train.shape}")
print(f"Testing sebelum reduksi : {df_test.shape}")

# Pisahkan kolom fitur (tanpa ID dan Label)
fitur_train = df_train.drop(columns=['ID', 'Label']).values
fitur_test = df_test.drop(columns=['ID', 'Label']).values

# Terapkan SVD
svd = TruncatedSVD(n_components=N_KOMPONEN, random_state=42)
svd_train = svd.fit_transform(fitur_train)
svd_test = svd.transform(fitur_test)

variance = sum(svd.explained_variance_ratio_) * 100
print(f"\nVariance explained: {variance:.2f}%")

# Buat DataFrame hasil: ID | SVD1 | SVD2 | ... | SVD200 | Label
kolom_svd = [f'SVD{i+1}' for i in range(N_KOMPONEN)]

df_svd_train = pd.DataFrame(svd_train, columns=kolom_svd)
df_svd_train.insert(0, 'ID', df_train['ID'].values)
df_svd_train['Label'] = df_train['Label'].values

df_svd_test = pd.DataFrame(svd_test, columns=kolom_svd)
df_svd_test.insert(0, 'ID', df_test['ID'].values)
df_svd_test['Label'] = df_test['Label'].values

print(f"Training sesudah reduksi: {df_svd_train.shape}")
print(f"Testing sesudah reduksi : {df_svd_test.shape}")

# Simpan
df_svd_train.to_csv("tfidf_svd_training.csv", index=False)
df_svd_train.to_excel("tfidf_svd_training.xlsx", index=False)
print(f"\nTersimpan: tfidf_svd_training.csv / .xlsx")

df_svd_test.to_csv("tfidf_svd_testing.csv", index=False)
df_svd_test.to_excel("tfidf_svd_testing.xlsx", index=False)
print(f"Tersimpan: tfidf_svd_testing.csv / .xlsx")


Training sebelum reduksi: (160, 7426)
Testing sebelum reduksi : (40, 7426)

Variance explained: 100.00%
Training sesudah reduksi: (160, 162)
Testing sesudah reduksi : (40, 162)

Tersimpan: tfidf_svd_training.csv / .xlsx
Tersimpan: tfidf_svd_testing.csv / .xlsx
